## Censo universitarios 1996 

In [85]:
# Paquetes
from pathlib import Path

import unicodedata
import pandas as pd
import pyreadstat
import matplotlib.pyplot as plt
import seaborn as sns

In [86]:
# Rutas de Censo 1996
ruta_1996 = Path(r"E:\BD\Censo Universitario\1996")
ruta_pregrado_1996 = ruta_1996 / "1 Estudiante de Pre-Grado"
ruta_posgrado_1996 = ruta_1996 / "2 Estudiante de Post-Grado"
ruta_universidades_1996 = ruta_1996 / "6 Universidad"

# Rutas de programas
ruta_correspondencias = Path(
    r"D:\Daniel_Pajita\1_Proyectos\Research\Educacion"
    r"\educacion-universitaria-peru\2_Datos"
    r"\Correspondencias_entre_clasificaciones"
)

#### Base de pregrado 1996

In [87]:
# Cargar los cinco módulos de pregrado
bases_pregrado_1996 = {}
metadatos_pregrado_1996 = {}

for modulo in ["M1S01", "M1S02", "M1S03", "M1S04", "M1S05"]:
    datos, meta = pyreadstat.read_sav(str(ruta_pregrado_1996 / f"{modulo}.sav"))
    datos["key"] = (datos["key"].astype("string").str.strip().replace("", pd.NA))

    bases_pregrado_1996[modulo] = datos
    metadatos_pregrado_1996[modulo] = meta

# Cargar la base de universidades
universidades_1996, meta_universidades_1996 = pyreadstat.read_sav(str(ruta_universidades_1996 / "M6S01.sav"))
universidades_1996.columns = universidades_1996.columns.str.lower()

In [88]:
# Bases de pregrado
m1s01 = bases_pregrado_1996["M1S01"]
m1s02 = bases_pregrado_1996["M1S02"]
m1s03 = bases_pregrado_1996["M1S03"]
m1s04 = bases_pregrado_1996["M1S04"]
m1s05 = bases_pregrado_1996["M1S05"]

In [89]:
# Revisar duplicados en cada módulo
for nombre, base in [
    ("M1S01", m1s01),
    ("M1S02", m1s02),
    ("M1S03", m1s03),
    ("M1S04", m1s04),
    ("M1S05", m1s05),
]:
    print(f"{nombre}: {base['key'].duplicated().sum():,} duplicados")

M1S01: 14 duplicados
M1S02: 3 duplicados
M1S03: 3 duplicados
M1S04: 3 duplicados
M1S05: 3 duplicados


In [90]:
# Preparar los módulos y resolver duplicados
modulos = {}
registros_descartados = {}

for modulo, base in bases_pregrado_1996.items():
    datos = base.dropna(subset=["key"]).drop_duplicates().copy()
    datos["_completos"] = (datos.drop(columns="key").replace(r"^\s*$", pd.NA, regex=True).notna().sum(axis=1))
    datos = datos.sort_values("_completos", ascending=False, kind="stable")
    descartar = datos["key"].duplicated(keep="first")
    registros_descartados[modulo] = (datos.loc[descartar].drop(columns="_completos"))
    
    modulos[modulo] = (
        datos.loc[~descartar].drop(columns="_completos").rename(columns={columna: f"{modulo}_{columna}"
            for columna in base.columns
            if columna != "key"
        })
    )

# Conservar las coincidencias entre M1S01 y M1S02
pregrado_1996 = modulos["M1S01"].merge(modulos["M1S02"],on="key",how="inner",validate="one_to_one",)

# Incorporar los otros módulos sin reducir la población inicial
for modulo in ["M1S03", "M1S04", "M1S05"]:
    pregrado_1996 = pregrado_1996.merge(
        modulos[modulo],
        on="key",
        how="left",
        validate="one_to_one",
    )
pregrado_1996.insert(0, "id_fila", range(1, len(pregrado_1996) + 1))

print(f"Base integrada de pregrado: {len(pregrado_1996):,} registros")

Base integrada de pregrado: 335,711 registros


In [91]:
# Seleccionar las variables para preparar la base de análisis
variables_pregrado = {
    "id_fila": "id_fila",
    "key": "key",
    "M1S01_p3": "codigo_universidad",
    "M1S01_p4": "codigo_facultad",
    "M1S01_p5": "codigo_programa",
    "M1S01_p5_a": "nombre_programa",
    "M1S02_p6": "sexo",
    "M1S02_p7_1": "dia_nacimiento",
    "M1S02_p7_2": "mes_nacimiento",
    "M1S02_p7_3": "anio_nacimiento_original",
    "M1S02_p9": "codigo_nacionalidad",
    "M1S01_p2": "ubigeo_original",
    "M1S01_p2_nomdpto": "departamento_original",
    "M1S01_p2_nomprov": "provincia_original",
    "M1S01_p2_nomdist": "distrito_original",
    "M1S02_p8": "codigo_lugar_nacimiento",
}

pregrado_1996_analisis = (pregrado_1996[list(variables_pregrado)].rename(columns=variables_pregrado).copy())
pregrado_1996_analisis["nivel_academico"] = "Pregrado"

# Fecha de nacimiento y edad al cierre del empadronamiento
fecha_referencia = pd.Timestamp("1996-07-15")

anio = pd.to_numeric(pregrado_1996_analisis["anio_nacimiento_original"],errors="coerce",)

pregrado_1996_analisis["fecha_nacimiento"] = pd.to_datetime(
    {
        "year": 1900 + anio.where(anio.between(16, 82)),
        "month": pregrado_1996_analisis["mes_nacimiento"],
        "day": pregrado_1996_analisis["dia_nacimiento"],
    },
    errors="coerce",
)

nacimiento = pregrado_1996_analisis["fecha_nacimiento"]
cumple_despues = (
    (nacimiento.dt.month > fecha_referencia.month) | ((nacimiento.dt.month == fecha_referencia.month) & (nacimiento.dt.day > fecha_referencia.day))
)

pregrado_1996_analisis["edad"] = (fecha_referencia.year - nacimiento.dt.year - cumple_despues.astype(int)).astype("Int64")


# Preparar el catálogo de universidades
catalogo_universidades_1996 = (
    universidades_1996[["p1_1", "p1_2", "p2"]]
    .rename(columns={"p1_1": "codigo_universidad", "p1_2": "nombre_universidad", "p2": "tipo_universidad",})
    .copy()
)

catalogo_universidades_1996["nombre_universidad"] = (catalogo_universidades_1996["nombre_universidad"].str.strip())

catalogo_universidades_1996["tipo_universidad"] = (
    catalogo_universidades_1996["tipo_universidad"].map({
        1: "Pública",
        2: "Privada",
    }).fillna("No especifica")
)

# Incorporar nombre y tipo de universidad
pregrado_1996_analisis = pregrado_1996_analisis.merge(
    catalogo_universidades_1996,
    on="codigo_universidad",
    how="left",
    validate="many_to_one",
)

In [92]:
pregrado_1996_analisis.head()

,id_fila,key,codigo_universidad,codigo_facultad,codigo_programa,nombre_programa,sexo,dia_nacimiento,mes_nacimiento,anio_nacimiento_original,...,ubigeo_original,departamento_original,provincia_original,distrito_original,codigo_lugar_nacimiento,nivel_academico,fecha_nacimiento,edad,nombre_universidad,tipo_universidad
0,1,1421 546,1.0,421.0,1.0,PSICOLOGIA,1.0,19.0,12.0,75.0,...,150101,LIMA,LIMA,LIMA,150113.0,Pregrado,1975-12-19,20,UNIVERSIDAD NACIONAL MAYOR DE SAN MARCOS,Pública
1,2,1421 550,1.0,421.0,1.0,PSICOLOGIA,1.0,0.0,0.0,0.0,...,150132,LIMA,LIMA,SAN JUAN DE LURIGANCHO,190199.0,Pregrado,NaT,<NA>,UNIVERSIDAD NACIONAL MAYOR DE SAN MARCOS,Pública
2,3,2412 1010,2.0,412.0,4.0,EDUCACION FISICA,2.0,16.0,10.0,72.0,...,050101,AYACUCHO,HUAMANGA,AYACUCHO,50101.0,Pregrado,1972-10-16,23,UNIVERSIDAD NACIONAL SAN CRISTOBAL DE HUAMANGA,Pública
3,4,2414 36,2.0,414.0,2.0,ARQUEOLOGIA E HISTORIA,1.0,7.0,10.0,72.0,...,050101,AYACUCHO,HUAMANGA,AYACUCHO,51108.0,Pregrado,1972-10-07,23,UNIVERSIDAD NACIONAL SAN CRISTOBAL DE HUAMANGA,Pública
4,5,2414 37,2.0,414.0,2.0,ARQUEOLOGIA E HISTORIA,2.0,24.0,10.0,68.0,...,050101,AYACUCHO,HUAMANGA,AYACUCHO,50101.0,Pregrado,1968-10-24,27,UNIVERSIDAD NACIONAL SAN CRISTOBAL DE HUAMANGA,Pública


In [93]:
# Edad promedio por sexo y tipo de universidad: pregrado
tabla_edad_pregrado = pd.pivot_table(
    pregrado_1996_analisis.assign(Sexo=pregrado_1996_analisis["sexo"].map({1: "Masculino",2: "Femenino",})),
    values="edad",
    index="Sexo",
    columns="tipo_universidad",
    aggfunc="mean",
    margins=True,
    margins_name="TOTAL",
)

display(tabla_edad_pregrado.style.format("{:.2f}", na_rep="—"))

tipo_universidad,Privada,Pública,TOTAL
Sexo,,,
Femenino,21.61,22.08,21.87
Masculino,22.50,23.29,23.00
TOTAL,22.04,22.78,22.49


#### Base de posgrado 1996

In [94]:
# Cargar los cinco módulos de posgrado
bases_posgrado_1996 = {}
metadatos_posgrado_1996 = {}

for modulo in ["M2S01", "M2S02", "M2S03", "M2S04", "M2S05"]:
    datos, meta = pyreadstat.read_sav(str(ruta_posgrado_1996 / f"{modulo}.sav"))
    datos["key"] = (datos["key"].astype("string").str.strip().replace("", pd.NA))

    bases_posgrado_1996[modulo] = datos
    metadatos_posgrado_1996[modulo] = meta

In [95]:
# Bases de posgrado
m2s01 = bases_posgrado_1996["M2S01"]
m2s02 = bases_posgrado_1996["M2S02"]
m2s03 = bases_posgrado_1996["M2S03"]
m2s04 = bases_posgrado_1996["M2S04"]
m2s05 = bases_posgrado_1996["M2S05"]

In [96]:
# Revisar duplicados en cada módulo
for nombre, base in [
    ("M2S01", m2s01),
    ("M2S02", m2s02),
    ("M2S03", m2s03),
    ("M2S04", m2s04),
    ("M2S05", m2s05),
]:
    print(f"{nombre}: {base['key'].duplicated().sum():,} duplicados")

M2S01: 0 duplicados
M2S02: 0 duplicados
M2S03: 11,203 duplicados
M2S04: 1,461 duplicados
M2S05: 0 duplicados


In [97]:
# Emparejar identificación y características generales de posgrado
posgrado_1996 = (
    m2s01.dropna(subset=["key"])
    .rename(columns={
        columna: f"M2S01_{columna}"
        for columna in m2s01.columns
        if columna != "key"
    })
    .merge(
        m2s02.dropna(subset=["key"]).rename(columns={
            columna: f"M2S02_{columna}"
            for columna in m2s02.columns
            if columna != "key"
        }),
        on="key",
        how="inner",
        validate="one_to_one",
    )
)

posgrado_1996.insert(0, "id_fila", range(1, len(posgrado_1996) + 1))

print(f"Base integrada de posgrado: {len(posgrado_1996):,} registros")

Base integrada de posgrado: 10,818 registros


In [98]:
# Seleccionar las variables de análisis de posgrado
variables_posgrado = {
    "id_fila": "id_fila",
    "key": "key",
    "M2S01_p3": "codigo_universidad",
    "M2S01_p4": "codigo_facultad",
    "M2S01_p4_a": "codigo_nivel_academico",
    "M2S01_p4_b": "codigo_programa",
    "M2S01_p4_c": "nombre_programa",
    "M2S02_p5": "sexo",
    "M2S02_p6_1": "dia_nacimiento",
    "M2S02_p6_2": "mes_nacimiento",
    "M2S02_p6_3": "anio_nacimiento_original",
    "M2S02_p8": "codigo_nacionalidad",
    "M2S01_p2": "ubigeo_original",
    "M2S01_p2_nomdpto": "departamento_original",
    "M2S01_p2_nomprov": "provincia_original",
    "M2S01_p2_nomdist": "distrito_original",
    "M2S02_p7": "codigo_lugar_nacimiento",
}

posgrado_1996_analisis = (posgrado_1996[list(variables_posgrado)].rename(columns=variables_posgrado).copy())

# Fecha de nacimiento y edad al cierre del empadronamiento
fecha_referencia = pd.Timestamp("1996-07-15")

anio = pd.to_numeric(posgrado_1996_analisis["anio_nacimiento_original"],errors="coerce",)

posgrado_1996_analisis["fecha_nacimiento"] = pd.to_datetime(
    {
        "year": 1900 + anio.where(anio.between(16, 77)),
        "month": posgrado_1996_analisis["mes_nacimiento"],
        "day": posgrado_1996_analisis["dia_nacimiento"],
    },
    errors="coerce",
)

nacimiento = posgrado_1996_analisis["fecha_nacimiento"]

cumple_despues = (
    (nacimiento.dt.month > fecha_referencia.month) | ((nacimiento.dt.month == fecha_referencia.month)& (nacimiento.dt.day > fecha_referencia.day))
)
posgrado_1996_analisis["edad"] = (fecha_referencia.year - nacimiento.dt.year - cumple_despues.astype(int)).astype("Int64")


# Identificar el nivel académico
posgrado_1996_analisis["nivel_academico"] = (
    posgrado_1996_analisis["codigo_nivel_academico"].map({
        1: "Segunda especialidad",
        2: "Maestría",
        3: "Doctorado",
    }).fillna("No especifica")
)

# Incorporar nombre y tipo de universidad
posgrado_1996_analisis = posgrado_1996_analisis.merge(catalogo_universidades_1996,on="codigo_universidad",how="left",validate="many_to_one",)

In [99]:
posgrado_1996_analisis.head()

,id_fila,key,codigo_universidad,codigo_facultad,codigo_nivel_academico,codigo_programa,nombre_programa,sexo,dia_nacimiento,mes_nacimiento,...,ubigeo_original,departamento_original,provincia_original,distrito_original,codigo_lugar_nacimiento,fecha_nacimiento,edad,nivel_academico,nombre_universidad,tipo_universidad
0,1,1415 1,1.0,415.0,2.0,122.0,MATEMATICAS,1.0,10.0,10.0,...,150135,LIMA,LIMA,SAN MARTIN DE PORRES,150136.0,1951-10-10,44,Maestría,UNIVERSIDAD NACIONAL MAYOR DE SAN MARCOS,Pública
1,2,1415 2,1.0,415.0,2.0,122.0,MATEMATICAS,2.0,20.0,12.0,...,070101,CALLAO,CALLAO,CALLAO,150113.0,1968-12-20,27,Maestría,UNIVERSIDAD NACIONAL MAYOR DE SAN MARCOS,Pública
2,3,1415 3,1.0,415.0,2.0,76.0,ESTADISTICA,2.0,12.0,3.0,...,150142,LIMA,LIMA,VILLA EL SALVADOR,150509.0,1950-03-12,46,Maestría,UNIVERSIDAD NACIONAL MAYOR DE SAN MARCOS,Pública
3,4,1415 4,1.0,415.0,2.0,43.0,COMPUTACION E INFORMATICA,2.0,14.0,11.0,...,150117,LIMA,LIMA,LOS OLIVOS,150113.0,1965-11-14,30,Maestría,UNIVERSIDAD NACIONAL MAYOR DE SAN MARCOS,Pública
4,5,1415 5,1.0,415.0,2.0,122.0,MATEMATICAS,1.0,6.0,5.0,...,150101,LIMA,LIMA,LIMA,140308.0,1950-05-06,46,Maestría,UNIVERSIDAD NACIONAL MAYOR DE SAN MARCOS,Pública


In [100]:
# Edad promedio por sexo, tipo de universidad y nivel académico: posgrado
tabla_edad_posgrado = pd.pivot_table(
    posgrado_1996_analisis.assign(
        Sexo=posgrado_1996_analisis["sexo"].map({
            1: "Masculino",
            2: "Femenino",
        })
    ),
    values="edad",
    index=["nivel_academico", "Sexo"],
    columns="tipo_universidad",
    aggfunc="mean",
    margins=True,
    margins_name="TOTAL",
)

display(tabla_edad_posgrado.style.format("{:.2f}", na_rep="—"))

In [101]:
# Tabla resumen de pregrado y posgrado
datos_tabla = pd.concat(
    [
        pregrado_1996_analisis[["nivel_academico", "sexo", "tipo_universidad", "edad"]],
        posgrado_1996_analisis[["nivel_academico", "sexo", "tipo_universidad", "edad"]],
    ],
    ignore_index=True,
)
datos_tabla["Sexo"] = datos_tabla["sexo"].map({1: "Masculino", 2: "Femenino",})
niveles = ["Pregrado", "Segunda especialidad", "Maestría", "Doctorado"]
datos_tabla = datos_tabla.loc[datos_tabla["nivel_academico"].isin(niveles)]

# Resumir cada combinación, incluyendo los totales
resultados = {}

for sexo in ["Masculino", "Femenino", "TOTAL"]:
    for tipo in ["Pública", "Privada", "TOTAL"]:
        grupo = datos_tabla

        if sexo != "TOTAL":
            grupo = grupo.loc[grupo["Sexo"] == sexo]
        if tipo != "TOTAL":
            grupo = grupo.loc[grupo["tipo_universidad"] == tipo]

        resumen = (
            grupo.groupby("nivel_academico")
            .agg(Edad_promedio=("edad", "mean"),Estudiantes=("edad", "size"),)
            .reindex(niveles)
        )

        resumen.loc["TOTAL"] = [grupo["edad"].mean(), len(grupo),]
        resultados[(sexo, tipo)] = resumen

tabla_edad_nivel = pd.concat(resultados, axis=1)
tabla_edad_nivel.columns.names = ["Sexo", "Tipo de universidad", "Medida"]
tabla_edad_nivel.index.name = "Nivel académico"

display(tabla_edad_nivel.style.format(
    {
        columna: ("{:.1f}" if columna[2] == "Edad_promedio" else "{:,.0f}")
        for columna in tabla_edad_nivel.columns
    },
    na_rep="—",
))

### Programas del censo 1996

In [102]:
# Programas únicos de pregrado
programas_pregrado_1996 = (
    pregrado_1996_analisis[["nombre_programa", "nivel_academico"]]
    .drop_duplicates()
    .sort_values(["nivel_academico", "nombre_programa"])
    .reset_index(drop=True)
)

# Programas únicos de posgrado
programas_posgrado_1996 = (
    posgrado_1996_analisis[["nombre_programa", "nivel_academico"]]
    .drop_duplicates()
    .sort_values(["nivel_academico", "nombre_programa"])
    .reset_index(drop=True)
)

In [103]:
programas_pregrado_1996.head()

,nombre_programa,nivel_academico
0,,Pregrado
1,EDUCACION FISICA,Pregrado
2,40101 1MEDICINA HUMANA,Pregrado
3,40199 2PERIODISMO,Pregrado
4,EDUCACION FISICA,Pregrado


In [104]:
programas_posgrado_1996.head()

,nombre_programa,nivel_academico
0,CIENCIAS ADMINISTRATIVAS,Doctorado
1,CIENCIAS BIOLOGICAS,Doctorado
2,CIENCIAS CONTABLES,Doctorado
3,CIENCIAS ECONOMICAS,Doctorado
4,DERECHO,Doctorado


In [105]:
# Combinar los listados de programas
programas_1996 = pd.concat([programas_pregrado_1996, programas_posgrado_1996],ignore_index=True,)

# Estandarizar los nombres
def estandarizar_programa(texto):
    if pd.isna(texto):
        return pd.NA

    texto = unicodedata.normalize("NFD", str(texto).upper())
    texto = "".join(
        caracter for caracter in texto
        if unicodedata.category(caracter) != "Mn"
    )
    texto = "".join(
        caracter if caracter.isalnum() else " "
        for caracter in texto
    )
    return " ".join(texto.split())


programas_1996["nombre_programa"] = (programas_1996["nombre_programa"].map(estandarizar_programa))

# Conservar combinaciones únicas de programa y nivel
programas_1996 = (
    programas_1996
    .drop_duplicates(subset=["nombre_programa", "nivel_academico"])
    .sort_values(["nivel_academico", "nombre_programa"])
    .reset_index(drop=True)
)

# Exportar el insumo para clasificación
programas_1996.to_csv(ruta_correspondencias / "programas_censo1996.txt",sep="|",index=False,encoding="utf-8-sig",)

In [106]:
programas_1996

,nombre_programa,nivel_academico
0,CIENCIAS ADMINISTRATIVAS,Doctorado
1,CIENCIAS BIOLOGICAS,Doctorado
2,CIENCIAS CONTABLES,Doctorado
3,CIENCIAS ECONOMICAS,Doctorado
4,DERECHO,Doctorado
...,...,...
978,TECNOLOGIA EN DIDACTICA UNIVERSITARIA,Segunda especialidad
979,TOXICOLOGIA Y QUIMICA LEGAL,Segunda especialidad
980,TOXICOLOGIA Y QUIMICA LEGAL SIONAL,Segunda especialidad
981,TRAUMATOLOGIA ORTOPEDIA,Segunda especialidad


#### Crear BD del Censo 1996

In [107]:
ruta_salida_1996 = Path(
    r"D:\Daniel_Pajita\1_Proyectos\Research\Educacion"
    r"\educacion-universitaria-peru\2_Datos\Censo_1996"
)

salida = ruta_correspondencias / "BD_programas_prediccion_clasificada.txt"

In [ ]:
# Cargar la clasificación de programas
clasificacion = pd.read_csv(salida,sep="|",dtype="string",encoding="utf-8-sig",)

variables_grupos = ["codigo_grupo_1","nombre_grupo_1","codigo_grupo_3","nombre_grupo_3",]

# Crear las mismas claves de emparejamiento en ambas fuentes
def claves_programa(base):
    base = base.copy()

    base["_programa"] = (base["nombre_programa"].map(estandarizar_programa))
    base["_nivel"] = (base["nivel_academico"].map(estandarizar_programa).replace({"CARRERA PROFESIONAL": "PREGRADO"}))

    return base


clasificacion = claves_programa(clasificacion)
catalogo = (clasificacion[["_programa", "_nivel"] + variables_grupos].drop_duplicates())

# Evitar emparejar nombres o niveles vacíos
catalogo = catalogo.loc[
    catalogo["_programa"].notna()
    & catalogo["_programa"].ne("")
    & catalogo["_nivel"].notna()
    & catalogo["_nivel"].ne("")
]

# Incorporar los grupos sin perder registros ni alterar sus conteos
def incorporar_grupos(base):
    return (
        claves_programa(base)
        .merge(catalogo,on=["_programa", "_nivel"],how="left",validate="many_to_one",indicator="resultado_clasificacion",)
        .drop(columns=["_programa", "_nivel"])
    )

pregrado_1996_colapsado_clasificado = incorporar_grupos(pregrado_1996_colapsado)
posgrado_1996_colapsado_clasificado = incorporar_grupos(posgrado_1996_colapsado)

In [109]:
# Exportar las bases completas: Censo 1996
pregrado_1996_colapsado_clasificado.to_csv(ruta_salida_1996 / "BD_pregrado_1996.txt",sep="|",index=False,encoding="utf-8-sig",)
posgrado_1996_colapsado_clasificado.to_csv(ruta_salida_1996 / "BD_posgrado_1996.txt",sep="|",index=False,encoding="utf-8-sig",)

In [111]:
# Frecuencia y porcentaje por grupo de carreras: nivel 1
tabla_grupos_pregrado = (
    pregrado_1996_colapsado_clasificado
    .assign(
        Grupo=lambda df: df["nombre_grupo_1"]
        .fillna("Sin clasificación")
        .replace(r"^\s*$", "Sin clasificación", regex=True)
    )
    .groupby("Grupo")["numero_registros"]
    .sum()
    .sort_values(ascending=False)
    .to_frame("Registros")
)

tabla_grupos_pregrado["Porcentaje"] = (
    tabla_grupos_pregrado["Registros"]
    / tabla_grupos_pregrado["Registros"].sum()
    * 100
)

tabla_grupos_pregrado.loc["TOTAL"] = tabla_grupos_pregrado.sum()
display(tabla_grupos_pregrado.style.format({"Registros": "{:,.0f}","Porcentaje": "{:.2f} %",}))

,Registros,Porcentaje
Grupo,,
Ciencias Administrativas y Derecho,"81,985",24.42 %
Salud y bienestar,"52,544",15.65 %
"Ingeniería, Industria y Construcción","51,885",15.46 %
Educación,"41,425",12.34 %
"Ciencias Sociales, Periodismo e Información","35,025",10.43 %
"Agricultura, Silvicultura, Pesca y Veterinaria","26,123",7.78 %
"Ciencias Naturales, Matemáticas y Estadística","16,980",5.06 %
Tecnología de la Información y la Comunicación,"14,449",4.30 %
Arte y Humanidades,"11,342",3.38 %


In [ ]:
# Frecuencia y porcentaje por grupo de carreras: nivel 3
tabla_grupos_pregrado = (
    pregrado_1996_colapsado_clasificado
    .assign(
        Grupo=lambda df: df["nombre_grupo_3"]
        .fillna("Sin clasificación")
        .replace(r"^\s*$", "Sin clasificación", regex=True)
    )
    .groupby("Grupo")["numero_registros"]
    .sum()
    .sort_values(ascending=False)
    .to_frame("Registros")
)

tabla_grupos_pregrado["Porcentaje"] = (
    tabla_grupos_pregrado["Registros"]
    / tabla_grupos_pregrado["Registros"].sum()
    * 100
)

tabla_grupos_pregrado.loc["TOTAL"] = tabla_grupos_pregrado.sum()
display(tabla_grupos_pregrado.style.format({"Registros": "{:,.0f}","Porcentaje": "{:.2f} %",}))

,Registros,Porcentaje
Grupo,,
Derecho Público,"29,593",8.82 %
Contabilidad e Impuestos,"27,137",8.08 %
Gestión y Administración,"22,165",6.60 %
Odontología,"20,230",6.03 %
Producción Agrícola y Ganadera,"19,149",5.70 %
Economía,"14,489",4.32 %
Otros Programas en Educación,"14,124",4.21 %
Medicina,"12,312",3.67 %
Construcción e Ingeniería Civil,"11,017",3.28 %


In [112]:
# Frecuencia y porcentaje por grupo de carreras: posgrado, nivel 1
tabla_grupos_posgrado = (
    posgrado_1996_colapsado_clasificado
    .assign(
        Grupo=lambda df: df["nombre_grupo_1"]
        .fillna("Sin clasificación")
        .replace(r"^\s*$", "Sin clasificación", regex=True)
    )
    .groupby("Grupo")["numero_registros"]
    .sum()
    .sort_values(ascending=False)
    .to_frame("Registros")
)

tabla_grupos_posgrado["Porcentaje"] = (
    tabla_grupos_posgrado["Registros"]
    / tabla_grupos_posgrado["Registros"].sum() * 100
)

tabla_grupos_posgrado.loc["TOTAL"] = tabla_grupos_posgrado.sum()

display(tabla_grupos_posgrado.style.format({
    "Registros": "{:,.0f}",
    "Porcentaje": "{:.2f} %",
}))

,Registros,Porcentaje
Grupo,,
Ciencias Administrativas y Derecho,"3,344",30.91 %
Salud y bienestar,"2,516",23.26 %
Educación,"1,752",16.20 %
"Ciencias Sociales, Periodismo e Información",861,7.96 %
"Ciencias Naturales, Matemáticas y Estadística",565,5.22 %
Arte y Humanidades,442,4.09 %
Servicios,401,3.71 %
Tecnología de la Información y la Comunicación,358,3.31 %
"Ingeniería, Industria y Construcción",347,3.21 %
